# SE2: Data Processing — HydroReg-Oulujoki

This notebook converts raw data downloaded by `SE1_data_access.ipynb` into analysis-ready datasets for the HydroReg-Oulujoki modelling pipeline. It reads from `../HydroReg-Oulujoki_rawdata/` and writes to `../HydroReg-Oulujoki_analysis_ready/`.

**This notebook focuses on data processing only.** It does not run HBV-light calibration, lake inflow reconstruction, or FlexTool scheduling.


# Generating analysis ready data

**Definitions**

- **Raw data**: unpublished data simulated or collected by the researcher, or published data (any processing level) with metadata and a stable URL or persistent identifier. Here: Syke spatial datasets, CLC 2018 land cover, Syke hydrology API responses, FMI WFS XML responses, plant characteristic web documents.
- **Analysis-ready data**: datasets constructed from raw data that have been preprocessed, quality-controlled, standardized, and harmonized across space and time, and formatted into software-compatible objects so they can be directly used for analysis, modeling, and decision-making. Here: basin/sub-catchment polygons (ETRS-TM35FIN), HRU fraction tables, daily forcing time series (P, T, PET, Q_obs) for HBV-light, daily inflow ensembles at FlexTool tributary nodes, hourly forcing bundles for FlexTool.

**Modular workflow**:
1. Data ingestion — load from `../HydroReg-Oulujoki_rawdata/`
2. Quality control — flag outliers, check physical plausibility, document uncertainty sources
3. Harmonization — reproject to EPSG:3067, clip to Oulujoki basin, align to daily timestep
4. Variable derivation — compute HRU fractions, PET, areal met, FlexTool inflow bundles
5. Export — write to `../HydroReg-Oulujoki_analysis_ready/` with provenance sidecar

**Manuscript alignment**: this notebook translates nearly line-by-line into the Methods section. Note sources of error, bias, and uncertainty in all input datasets; document how each processing step interacts with error; and in the Discussion note whether each step reduces information content or introduces new sources of error, bias, or uncertainty.

**GUI-based software**: if any processing step requires a GUI tool, document every operation and all options selected in a markdown cell here.

Always check the CRS of every dataset. Plot and visually confirm correct projections before automating. Automate batch processing over multiple files; never copy-paste code.


In [14]:
import geopandas as gpd
import fiona
import pandas as pd
import numpy as np
from pathlib import Path
import logging
import json
import zipfile
import xml.etree.ElementTree as ET
from datetime import datetime
import sys

import rasterio
from rasterio.mask import mask as rasterio_mask
from rasterio.features import shapes
from rasterio.warp import calculate_default_transform, reproject, Resampling

from shapely.geometry import box, mapping
from shapely.ops import unary_union
import pyproj
from pyproj import CRS

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)

# Overview

## Input data description

All raw datasets are downloaded by `SE1_data_access.ipynb` and stored in `../HydroReg-Oulujoki_rawdata/`.

| Attribute | Syke catchments | CLC 2018 (20 m, FI) | CLC 2018 (25 ha, EU) | Syke hydrology API | FMI WFS observations |
|-----------|----------------|---------------------|----------------------|-------------------|----------------------|
| Name | valumaalueet | clc2018_fi20m | clc2018eu25ha | Paikka / Virtaama / Vedenkorkeus / Haihdunta | Hourly weather, Oulu |
| Source | Syke open data | Syke / CORINE | EEA / Syke | Hertta / Hydrologiarajapinta 1.1 | FMI Open Data WFS 2.0 |
| Raw file / endpoint | `direct_downloads/valumaalueet.zip` | `direct_downloads/clc2018_fi20m.zip` | `direct_downloads/clc2018eu25ha.zip` | `api/syke_*.json` | `api/fmi_oulu_*.xml` |
| Data type | Vector (polygon) | Raster | Raster | Tabular (JSON / OData) | Tabular (WML2 / XML) |
| CRS (native) | ETRS-TM35FIN (EPSG:3067) | ETRS-TM35FIN (EPSG:3067) | ETRS-LAEA (EPSG:3035) | — (attribute table) | WGS84 (EPSG:4326) |
| Spatial resolution | — | 20 m | 25 ha MMU | point / station | point / station |
| Temporal resolution | static | 2018 snapshot | 2018 snapshot | daily | hourly |
| Spatial extent | Finland | Finland | Europe | Oulujoki basin stations | Oulu region |
| Duration | — | — | — | 2014–2024 | 2014–2024 |
| Variable(s) | Sub-catchment polygons | Land-cover class | Land-cover class | Discharge (m³/s), water level (cm), evaporation (mm) | P (mm), T (°C), wind, RH |
| Encoding / format | Shapefile (zipped) | GeoTIFF (zipped) | GeoTIFF (zipped) | OData JSON | WML2 XML |
| Known errors / biases | — | Classification uncertainty ±1 class | Coarser MMU than 20 m product | Rating-curve uncertainty in Q | Point observation; spatial interpolation needed |

**Plant characteristics** (static, from public webpages — Fortum / Oulun Energia): head (m) and installed capacity (MW) per plant. These are entered manually in Section 4.4.

## Analysis-ready data description

| Attribute | Spatial layers | Daily hydro forcing | Hourly FlexTool bundle |
|-----------|---------------|--------------------|-----------------------|
| Data object(s) | GeoPackage (polygon) | Parquet + CSV | Parquet + CSV |
| CRS (EPSG) | 3067 | — | — |
| Spatial resolution | sub-catchment | — | — |
| Temporal resolution | static | daily | hourly |
| Spatial extent | Oulujoki basin | Oulujoki basin | Oulujoki cascade |
| Duration | — | 2014–2024 | 2014–2024 |
| Variable(s) and units | polygon geometry, catchment ID, area (km²), HRU fractions (urban / agri / forest / wetland / water, 0–1) | P (mm/day), T (°C), PET (mm/day), Q_obs (m³/s) per sub-catchment | Q_inflow (m³/s) per FlexTool node, price (€/MWh) |
| File format | `.gpkg` | `.parquet` + `.csv` | `.parquet` + `.csv` |

## Modelling framework

- **HBV-light** (Seibert & Vis, 2012): consumes daily P (mm/day), T (°C), and PET (mm/day) per sub-catchment and outputs daily discharge. Input format is whitespace-delimited `.txt`.
- **IRENA FlexTool** (rolling-horizon dispatch): consumes hourly inflow time series at cascade nodes (m³/s), hourly day-ahead electricity prices (€/MWh), reservoir storage capacities (Mm³), and turbine / spill constraints. Input format is CSV.

## Data processing steps

1. **(Section 1)** Load and inspect all raw datasets; confirm CRS, extents, and variable names.
2. **(Section 2)** QC discharge, water level, evaporation, and meteorological time series: flag missing values, apply physical plausibility bounds, identify duplicate records.
3. **(Section 3 — spatial)** Reproject all spatial datasets to EPSG:3067; clip to Oulujoki basin bounding box; compute catchment areas (km²).
4. **(Section 3 — raster)** Reproject CLC 2018 (20 m) to EPSG:3067 if needed; clip to basin; reclassify CORINE codes to urban / agriculture / forest / wetland / water.
5. **(Section 3 — time series)** Parse Syke OData JSON and FMI WML2 XML to tidy long-format DataFrames; align to a common daily time index (UTC+2); fill short gaps (≤ 3 days) by linear interpolation; flag longer gaps.
6. **(Section 4)** Derive PET from temperature using the Oudin (2005) method; aggregate point meteorological observations to sub-catchment means (Thiessen polygons or nearest-station assignment); disaggregate daily inflows to hourly by piecewise-constant repetition (volume-preserving) for FlexTool.
7. **(Section 4)** Compute operational envelope parameters (P5 and max of observed daily total outflow 2014–2024); enter plant characteristics (head, MW) from Fortum / Oulun Energia documentation.
8. **(Section 5)** Export all spatial and time-series outputs; write `provenance.json` sidecar.


## How to use this notebook:

### Prerequisites

Before running this notebook, make sure:

1. **`SE1_data_access.ipynb` has been run first.** SE2 reads everything from `../HydroReg-Oulujoki_rawdata/` which SE1 populates. If SE1 hasn't run, cells will fail on `assert` checks.
2. **Disk space** — run Cell 5 (`df -k ..`) to confirm you have enough space before unzipping the large CLC rasters (~1.5 GB).
3. **Dependencies** are installed: `geopandas`, `rasterio`, `pyproj`, `pandas`, `numpy`, `matplotlib`, `fiona`, `shapely`, `rasterstats`, `scipy`.

---

### Directory structure expected

```
parent/
├── HydroReg-Oulujoki_rawdata/
│   ├── direct_downloads/
│   │   ├── valumaalueet.zip       ← Syke catchments
│   │   ├── clc2018_fi20m.zip      ← CLC raster Finland 20m
│   │   └── clc2018eu25ha.zip      ← CLC raster EU 25ha
│   └── api/
│       ├── syke_Paikka.json
│       ├── syke_Virtaama.json
│       ├── syke_Vedenkorkeus.json
│       ├── syke_Haihdunta.json
│       └── fmi_oulu_*.xml
└── HydroReg-Oulujoki_analysis_ready/   ← created by this notebook
```

---

### Running the notebook — section by section

The notebook must be run **top to bottom**. Many cells are scaffolded with commented-out code — you need to uncomment and adapt them as you inspect your actual data.

**Section 1 — Data ingestion**
- Run Cell 7 first with just `fiona.listlayers()` to see the layer names in `valumaalueet.zip`, then uncomment `gpd.read_file()` with the correct layer name.
- Cell 8 inspects CLC zip contents — uncomment the `rasterio.open()` block once you've confirmed the `.tif` filename inside the zip.
- Cells 9–10 load Syke API JSON and FMI XML. Check that the column names printed match what SE1 actually downloaded — you may need to adjust `'Aika'`, `'Arvo'`, `'PaikkaId'` in Section 2.

**Section 2 — Quality control**
- All QC cells (2.1–2.3) are fully commented out. Uncomment them one at a time, adjust column names to match your SE1 output, and re-run.
- Document every flagged record count in the markdown — this is required for the Discussion.

**Section 3 — Harmonization**
- The bounding box `OULUJOKI_BBOX_3067` in Cell 16 is approximate — confirm it visually after plotting catchments in Cell 17 before enabling the clip.
- Cell 16 also sets the key parameters (`TARGET_CRS`, `START_DATE`, `END_DATE`, `GAP_INTERP_MAX_DAYS`) — adjust here if needed, they propagate to the provenance sidecar automatically.

**Section 4 — Variable derivation**
- Cell 25 sets `TURBINE_EFFICIENCY = 0.90` — update this if you have a more precise plant-specific value.
- Cell 23 uses `OULUJOKI_LATITUDE_DEG = 64.5` for PET — refine per sub-catchment centroid once catchment geometries are loaded.

**Section 5 — Export**
- Cells 5.1–5.3 are all commented out. Uncomment only after Sections 1–4 are producing valid outputs.
- Cell 5.4 (provenance) and Cell 5.5 (file inventory) run unconditionally — run them last to confirm what was actually written.

---

### Key things to customise before publishing

| What | Where | Action |
|---|---|---|
| Raw input filenames | Cell 31 (`provenance`) | Update `raw_input_files` with actual SE1 filenames |
| Catchment layer name | Cell 7 | Uncomment and set correct `layer=` |
| Oulujoki bounding box | Cell 16 | Confirm coordinates after visual check |
| CLC pixel value ranges | Cell 18 (`CLC_RECLASS`) | Verify with `np.unique(arr)` on actual raster |
| API column names | Cells 12–13 | Match to SE1 JSON/XML output |
| Sub-catchment latitude | Cell 23 | Refine per centroid for PET |

---

### Outputs

When complete, `../HydroReg-Oulujoki_analysis_ready/` will contain:

- `spatial/` — sub-catchment GeoPackage with HRU fractions + clipped CLC GeoTIFF
- `forcing/` — HBV-light daily forcing `.txt` per sub-catchment + observed discharge `.parquet`/`.csv`
- `flextool_inputs/` — hourly inflow `.parquet`/`.csv` per cascade node + plant characteristics `.csv`
- `provenance.json` — full processing metadata sidecar

## Storage requirements

| | Size (approx.) |
|--|----------------|
| Total output volume | ~50 MB (spatial + time series) |
| Largest individual file | clc2018_fi20m unzipped (~1.5 GB — processed in memory; clipped output << 100 MB) |


In [15]:
# Directory setup — automatically generated cell, do not modify
# Current working directory = project/notebooks/

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "README.md").exists():
            return p
    return start

notebook_dir  = Path.cwd()
repo_root     = find_repo_root(notebook_dir)
external_base = repo_root.parent

# Inputs: written by SE1_data_access.ipynb
raw_data_dir = external_base / "HydroReg-Oulujoki_rawdata"
direct_dir   = raw_data_dir  / "direct_downloads"
api_dir      = raw_data_dir  / "api"

# Outputs: analysis-ready datasets (mirrors processed_data/ in the repository structure)
ar_dir       = external_base / "HydroReg-Oulujoki_analysis_ready"
spatial_dir  = ar_dir / "spatial"
forcing_dir  = ar_dir / "forcing"
flextool_dir = ar_dir / "flextool_inputs"

for d in [spatial_dir, forcing_dir, flextool_dir]:
    d.mkdir(parents=True, exist_ok=True)

assert raw_data_dir.exists(), "Raw data directory not found — run SE1_data_access.ipynb first."

print(f"Notebook directory         : {notebook_dir}")
print(f"Repository root            : {repo_root}")
print(f"Raw data directory         : {raw_data_dir.resolve()}")
print(f"Analysis-ready directory   : {ar_dir.resolve()}")

Notebook directory         : /home/jovyan/WaterDig/ClassData
Repository root            : /home/jovyan/WaterDig
Raw data directory         : /home/jovyan/HydroReg-Oulujoki_rawdata
Analysis-ready directory   : /home/jovyan/HydroReg-Oulujoki_analysis_ready


In [16]:
# Check available storage before unzipping large raster files
!df -k ..
# Confirm available space exceeds total output volume stated in the Overview

Filesystem     1K-blocks    Used Available Use% Mounted on
/dev/sdb        10218772 5811132   4391256  57% /home/jovyan


---
# 1. Data ingestion

Load each raw dataset from `../HydroReg-Oulujoki_rawdata/`. Print CRS, extent, temporal range, and variable names for every dataset to confirm correct loading. Visually inspect all spatial projections before proceeding to Section 2.

In [17]:
# --- 1.1 Syke sub-catchments (valumaalueet) ---

catchments_zip = direct_dir / "valumaalueet.zip"
assert catchments_zip.exists(), f"Missing raw file: {catchments_zip}"

# Inspect layer names before loading
layers = fiona.listlayers(f"zip://{catchments_zip}")
print("Layers in valumaalueet.zip:", layers)

# Load — update layer name after inspecting output above
# catchments_raw = gpd.read_file(f"zip://{catchments_zip}", layer=layers[0])
# print("CRS         :", catchments_raw.crs)
# print("Columns     :", list(catchments_raw.columns))
# print("Total bounds:", catchments_raw.total_bounds)
# print("Feature count:", len(catchments_raw))
# catchments_raw.plot(figsize=(8, 8))
# plt.title("Syke catchments — native CRS visual check")
# plt.show()

Layers in valumaalueet.zip: ['Valumaaluejako_purkupiste', 'Valumaaluejako_taso1', 'Valumaaluejako_taso2', 'Valumaaluejako_taso3', 'Valumaaluejako_taso4', 'Valumaaluejako_taso5']


In [18]:
# --- 1.2 CLC 2018 rasters ---

clc_fi_zip = direct_dir / "clc2018_fi20m.zip"
clc_eu_zip = direct_dir / "clc2018eu25ha.zip"

for zp in [clc_fi_zip, clc_eu_zip]:
    assert zp.exists(), f"Missing raw file: {zp}"
    with zipfile.ZipFile(zp) as z:
        tifs = [n for n in z.namelist() if n.lower().endswith('.tif')]
        print(f"\n{zp.name}: {len(z.namelist())} files | GeoTIFFs: {tifs[:5]}")

# Open raster metadata without loading the full array
# with rasterio.open(f"zip+file://{clc_fi_zip}!/{tifs[0]}") as src:
#     print("CRS:", src.crs)
#     print("Resolution (m):", src.res)
#     print("Bounds:", src.bounds)
#     print("NoData:", src.nodata)


clc2018_fi20m.zip: 7 files | GeoTIFFs: ['Clc2018_FI20m.tif']

clc2018eu25ha.zip: 8 files | GeoTIFFs: []


In [19]:
# --- 1.3 Syke hydrology API responses (OData JSON) ---
# SE1 saves one JSON file per endpoint to api_dir/.
# Adjust filenames to match SE1 output.

syke_files = {
    'stations'    : api_dir / "syke_Paikka.json",
    'discharge'   : api_dir / "syke_Virtaama.json",
    'water_level' : api_dir / "syke_Vedenkorkeus.json",
    'evaporation' : api_dir / "syke_Haihdunta.json",
}

syke_raw = {}
for key, path in syke_files.items():
    if path.exists():
        with open(path) as f:
            data = json.load(f)
        records = data.get('value', data) 
        df = pd.json_normalize(records)
        syke_raw[key] = df
        print(f"{key:15s}: {len(df):6d} records | columns: {list(df.columns)[:6]}")
    else:
        log.warning(f"{key} not found at {path}")

stations       :   3966 records | columns: ['Paikka_Id', 'Suure_Id', 'H_Kunta_Id', 'KuntaNimi', 'Nro', 'Nimi']
discharge      : 125786 records | columns: ['Paikka_Id', 'Aika', 'Arvo', 'ArvoMin', 'ArvoMax', 'Lippu_id']
water_level    : 114059 records | columns: ['Paikka_Id', 'Aika', 'Arvo', 'ArvoMin', 'ArvoMax', 'Lippu_id']
evaporation    :   1128 records | columns: ['Paikka_Id', 'Aika', 'Arvo', 'Lippu_id']


In [20]:
# --- 1.4 FMI WFS XML responses (WML2) ---
# SE1 saves one XML file per API request to api_dir/.

fmi_files = sorted(api_dir.glob("fmi_oulu_*.xml"))
print(f"Found {len(fmi_files)} FMI XML file(s):")
for f in fmi_files:
    print(f"  {f.name}")

if fmi_files:
    tree = ET.parse(fmi_files[0])
    root = tree.getroot()
    print("Root tag        :", root.tag)
    print("numberReturned  :", root.attrib.get('numberReturned', '—'))
    print("timeStamp (UTC) :", root.attrib.get('timeStamp', '—'))

Found 1 FMI XML file(s):
  fmi_oulu_2024-01-01_2024-01-03_hourly.xml
Root tag        : {http://www.opengis.net/wfs/2.0}FeatureCollection
numberReturned  : 12
timeStamp (UTC) : 2026-03-20T10:39:36Z


---
# 2. Quality control

Detect and handle errors, biases, and outliers in each input dataset. Document every decision: threshold used, number of records flagged, treatment applied. Note residual uncertainty carried forward into the analysis-ready data.

**Expected uncertainty sources for this project:**
- Rating-curve uncertainty in Syke discharge records (especially at extreme flows — relevant for operational envelope derivation and HBV validation)
- Classification uncertainty in CLC 2018 (±1 class; MMU artefacts in the 25 ha EU product)
- Spatial representativeness of FMI point observations as sub-catchment areal estimates
- Data gaps in Syke time series (document gap length and treatment per station)

In [21]:
# --- 2.1 Discharge (Virtaama) QC ---

# Parse to tidy format
# q_raw = syke_raw['discharge'].copy()
# q_raw['datetime'] = pd.to_datetime(q_raw['Aika'])       # adjust column name to match API response
# q_raw = q_raw.rename(columns={'Arvo': 'Q_m3s', 'PaikkaId': 'station_id'})
# q_raw['Q_m3s'] = pd.to_numeric(q_raw['Q_m3s'], errors='coerce')

# Missing values
# print("Missing Q values:", q_raw['Q_m3s'].isna().sum())

# Physical plausibility: Q >= 0
# n_neg = (q_raw['Q_m3s'] < 0).sum()
# print(f"Negative Q values flagged: {n_neg}")
# q_raw.loc[q_raw['Q_m3s'] < 0, 'Q_m3s'] = np.nan

# Duplicates
# n_dup = q_raw.duplicated(subset=['station_id', 'datetime']).sum()
# print(f"Duplicate records removed: {n_dup}")
# q_raw = q_raw.drop_duplicates(subset=['station_id', 'datetime'])

# Gap audit per station
# for sid, grp in q_raw.groupby('station_id'):
#     s = grp.set_index('datetime').sort_index()['Q_m3s']
#     gaps = s.isna()
#     consec = (gaps * (gaps.groupby((~gaps).cumsum()).cumcount() + 1))
#     print(f"Station {sid}: {gaps.sum()} missing days | longest gap: {consec.max()} days")
pass

In [22]:
# --- 2.2 FMI meteorological observations QC ---

# Physical plausibility bounds (adjust based on FMI parameter names in the XML)
MET_BOUNDS = {
    'Precipitation amount' : (0,   200),   # mm/h
    'Air temperature'      : (-60,  45),   # °C
    'Relative humidity'    : (0,   100),   # %
    'Wind speed'           : (0,    80),   # m/s
}

# for var, (lo, hi) in MET_BOUNDS.items():
#     if var in met_df.columns:
#         mask = (met_df[var] < lo) | (met_df[var] > hi)
#         n = mask.sum()
#         print(f"{var}: {n} values outside [{lo}, {hi}] flagged")
#         met_df.loc[mask, var] = np.nan
pass

In [23]:
# --- 2.3 Spatial geometry validity check ---

# n_invalid = (~catchments_raw.is_valid).sum()
# print(f"Invalid geometries: {n_invalid}")
# if n_invalid > 0:
#     catchments_raw['geometry'] = catchments_raw.geometry.buffer(0)  # standard repair
#     print("Geometries repaired with buffer(0)")
pass

---
# 3. Harmonization

Align all datasets in space and time to a common CRS, spatial extent, and temporal resolution.

**Target CRS:** ETRS-TM35FIN `EPSG:3067` (native CRS of Syke catchments; Finnish standard)  
**Target spatial resolution:** sub-catchment polygon (vector); 20 m raster for CLC zonal statistics  
**Target temporal resolution:** daily (EET = UTC+2, aligned to 00:00–23:59 local time)  
**Study period:** 2015-01-01 to 2024-12-31  
**Spatial extent:** Oulujoki basin bounding box (clip all datasets to this extent)

In [24]:
TARGET_CRS  = 'EPSG:3067'
START_DATE  = '2015-01-01'
END_DATE    = '2024-12-31'
STUDY_DATES = pd.date_range(START_DATE, END_DATE, freq='D')
GAP_INTERP_MAX_DAYS = 3  # gaps <= this threshold filled by linear interpolation; longer gaps flagged

# Oulujoki basin bounding box in EPSG:3067 (approximate — confirm and adjust after visual check)
# OULUJOKI_BBOX_3067 = (300000, 7050000, 650000, 7300000)  # xmin, ymin, xmax, ymax

print(f"Target CRS    : {TARGET_CRS}")
print(f"Study period  : {START_DATE} — {END_DATE}")
print(f"Gap fill limit: {GAP_INTERP_MAX_DAYS} days (linear interpolation)")

Target CRS    : EPSG:3067
Study period  : 2015-01-01 — 2024-12-31
Gap fill limit: 3 days (linear interpolation)


In [25]:
# --- 3.1 Reproject catchments and clip to Oulujoki basin ---
# valumaalueet is natively EPSG:3067; confirm before skipping reprojection.

# catchments = catchments_raw.to_crs(TARGET_CRS)
# bbox_geom  = gpd.GeoDataFrame(geometry=[box(*OULUJOKI_BBOX_3067)], crs=TARGET_CRS)
# catchments = gpd.clip(catchments, bbox_geom)
# print(f"Sub-catchments after clip: {len(catchments)}")
# catchments.plot(figsize=(8, 8))
# plt.title("Oulujoki sub-catchments — EPSG:3067 visual check")
# plt.show()
pass

In [26]:
# --- 3.2 CLC 2018 raster — reproject (if needed) and clip to Oulujoki ---
# Use the 20 m Finnish product (EPSG:3067 native) for HRU fractions.
# The 25 ha EU product (EPSG:3035) is used as a secondary check only.

# CORINE land-cover reclassification (level-1 integer codes × 100 convention in FI product):
# Adjust code ranges after confirming actual values with: np.unique(arr)
CLC_RECLASS = {
    'urban'      : range(100, 200),
    'agriculture': range(200, 300),
    'forest'     : range(300, 400),
    'wetland'    : range(400, 500),
    'water'      : range(500, 600),
}

# Processing steps to implement:
# 1. Open raster via rasterio windowed read (avoid loading full ~1.5 GB array)
# 2. Mask (clip) to Oulujoki basin polygon
# 3. Reclassify pixels using CLC_RECLASS
# 4. Write clipped reclassified GeoTIFF to spatial_dir / 'clc2018_fi20m_oulujoki_reclass.tif'
print("CLC reclassification map defined. Implement after confirming native code values.")

CLC reclassification map defined. Implement after confirming native code values.


In [27]:
# --- 3.3 Time series harmonization ---

# All Syke data are reported in EET (UTC+2).
# FMI WML2 data are reported in UTC — convert to EET before daily aggregation.
#
# Daily aggregation rules:
#   Precipitation  : sum  (mm/day)
#   Temperature    : mean (°C)
#   Discharge      : mean (m³/s)
#   Water level    : mean (cm)
#   Evaporation    : sum  (mm/day)

def harmonize_daily(series: pd.Series, limit: int = GAP_INTERP_MAX_DAYS) -> pd.Series:
    """
    Reindex to STUDY_DATES, interpolate short gaps, flag remaining NaN.
    series : daily pd.Series with DatetimeIndex
    Returns: harmonized pd.Series aligned to STUDY_DATES
    """
    s = series.reindex(STUDY_DATES)
    s_filled = s.interpolate(method='linear', limit=limit)
    n_gap = s_filled.isna().sum()
    if n_gap > 0:
        log.warning(f"{series.name}: {n_gap} NaN remain after {limit}-day interpolation")
    return s_filled

print("harmonize_daily() defined. Apply to each discharge, met, and evaporation series.")

harmonize_daily() defined. Apply to each discharge, met, and evaporation series.


In [28]:
# --- 3.4 Unit standardization ---
# Document all unit conversions; apply to harmonized DataFrames.

# Example conversions (uncomment and adjust as needed):
# water_level_m = water_level_cm / 100          # cm → m
# precip_mm_day = precip_mm_h * 24              # mm/h hourly mean → mm/day (if aggregating hourly mean)

print("Unit conversions: document each one here with formula and source unit → target unit.")

Unit conversions: document each one here with formula and source unit → target unit.


---
# 4. Variable derivation

Convert harmonized inputs into the physically meaningful variables required by HBV-light and FlexTool. Domain expertise is applied here. For each derived variable, the mathematical definition, inputs consumed, assumptions made, and expected value range are documented in the cell below the code.

In [29]:
# --- 4.1 HRU area fractions per sub-catchment ---
# Inputs  : clipped CLC 2018 reclassified raster (Section 3.2), Oulujoki sub-catchment polygons
# Method  : zonal statistics — count reclassified pixels inside each sub-catchment polygon
# Units   : fraction [0, 1]; sum across classes must equal 1 per catchment (verified below)
# Assumptions: CLC 2018 land cover is representative of the study period (2014–2024)
# Uncertainty: CLC classification uncertainty (±1 level-1 class) affects HRU fractions at boundaries

# from rasterstats import zonal_stats
# clc_path = spatial_dir / 'clc2018_fi20m_oulujoki_reclass.tif'
# stats = zonal_stats(catchments, str(clc_path), categorical=True, nodata=-9999)
# hru_raw = pd.DataFrame(stats, index=catchments['catchment_id']).fillna(0)
# ... map integer class codes to labels, compute row-wise fractions ...

# Validation
# row_sums = hru_df[['urban','agriculture','forest','wetland','water']].sum(axis=1)
# assert ((row_sums - 1).abs() < 0.02).all(), "HRU fractions do not sum to ~1 — check NoData handling"
pass

In [30]:
# --- 4.2 Potential evapotranspiration (PET) — Oudin (2005) method ---
# Inputs  : daily mean air temperature T (°C), catchment centroid latitude (degrees)
# Formula : PET = Re / (lambda * rho_w) * (T + 5) / 100  if T + 5 > 0 else 0
#           where Re = extraterrestrial radiation (MJ/m²/day, derived from latitude and day of year)
#                 lambda = 2.45 MJ/kg (latent heat of vaporization)
#                 rho_w  = 1000 kg/m³ (water density)
# Units   : mm/day
# Assumptions: temperature is the primary control on PET at this latitude
# Uncertainty: ignores wind, humidity, and radiation variability — note in Discussion
# Reference: Oudin et al. (2005) doi:10.1016/j.jhydrol.2004.08.026

OULUJOKI_LATITUDE_DEG = 64.5  # approximate centroid; adjust per sub-catchment centroid

def oudin_pet(T_daily: pd.Series, lat_deg: float) -> pd.Series:
    """
    Oudin (2005) temperature-based PET.
    T_daily : daily mean air temperature (°C), DatetimeIndex
    lat_deg : catchment centroid latitude (degrees north)
    Returns : PET (mm/day)
    """
    lat_rad = np.radians(lat_deg)
    doy     = T_daily.index.dayofyear
    delta   = 0.409 * np.sin(2 * np.pi * doy / 365 - 1.39)
    dr      = 1 + 0.033 * np.cos(2 * np.pi * doy / 365)
    omega_s = np.arccos(-np.tan(lat_rad) * np.tan(delta))
    Re      = (24 / np.pi) * 0.082 * dr * (
        omega_s * np.sin(lat_rad) * np.sin(delta) +
        np.cos(lat_rad) * np.cos(delta) * np.sin(omega_s)
    )  # MJ/m²/day
    pet_mm  = np.where(T_daily + 5 > 0, (Re / (2.45 * 1000)) * ((T_daily + 5) / 100) * 1000, 0)
    return pd.Series(pet_mm, index=T_daily.index, name='PET_mm_day')

print("oudin_pet() defined. Apply to each sub-catchment temperature series.")

oudin_pet() defined. Apply to each sub-catchment temperature series.


In [31]:
# --- 4.3 Areal precipitation and temperature per sub-catchment ---
# Inputs  : FMI station observations (harmonized, daily), station locations (EPSG:3067),
#           sub-catchment polygons
# Method  : Thiessen polygon (Voronoi) area-weighted averaging
#   1. Compute Voronoi polygons from FMI station locations in EPSG:3067
#   2. Intersect Voronoi polygons with each sub-catchment polygon
#   3. Weight each station observation by the intersection area fraction
# Units   : mm/day (P), °C (T)
# Assumptions: linear spatial interpolation of station observations is appropriate;
#              station locations represent the surrounding Voronoi region
# Uncertainty: sparse station network — document number of stations and spatial coverage

# from scipy.spatial import Voronoi
# or use geopandas / shapely for Voronoi polygon construction
pass

In [32]:
# --- 4.4 Operational envelope parameters ---
# Inputs  : observed daily total outflow at Oulujoki outlet station, 2014–2024
# Q_min   : P5 (5th percentile) of observed daily total outflow
# Q_max   : maximum observed daily total outflow
# Turbine–generator efficiency: 0.90 (assumed constant; Quaranta et al., 2022)
# These parameters are used to define FlexTool turbine and spill feasibility constraints.

# q_outlet = ...  # daily discharge series (m³/s) at the outlet gauge
# Q_MIN = np.percentile(q_outlet.dropna(), 5)
# Q_MAX = q_outlet.max()
# print(f"Operational envelope: Q_min (P5) = {Q_MIN:.1f} m³/s | Q_max = {Q_MAX:.1f} m³/s")

TURBINE_EFFICIENCY = 0.90  # Quaranta et al. (2022)

# Plant characteristics — enter from Fortum and Oulun Energia public sources
# Source URLs:
#   Fortum    : https://www.fortum.com/energy-production/hydropower/plants/oulujoki-river-system
#   Oulun Energia: https://www.oulunenergia.fi/en/oulun-energia/energy-production/power-plants/

PLANT_DATA = pd.DataFrame([
    {'plant': 'Jylhämä',    'operator': 'Fortum',        'installed_MW': 55, 'head_m': 14},
    {'plant': 'Nuojua',     'operator': 'Fortum',        'installed_MW': 85, 'head_m': 22},
    {'plant': 'Utanen',     'operator': 'Fortum',        'installed_MW': 58.5, 'head_m': 15.7},
    {'plant': 'Pälli',      'operator': 'Fortum',        'installed_MW': 51, 'head_m': 14},
    {'plant': 'Pyhakoski',  'operator': 'Fortum',        'installed_MW': 147, 'head_m': 32.4},
    {'plant': 'Montta',     'operator': 'Fortum',        'installed_MW': 47, 'head_m': 12.2},
    {'plant': 'Merikoski',  'operator': 'Oulun Energia', 'installed_MW': 40, 'head_m': 11},
])
print("Plant data table initialized. Fill in installed_MW and head_m from source documents.")
print(PLANT_DATA.to_string(index=False))

Plant data table initialized. Fill in installed_MW and head_m from source documents.
    plant      operator  installed_MW  head_m
  Jylhämä        Fortum          55.0    14.0
   Nuojua        Fortum          85.0    22.0
   Utanen        Fortum          58.5    15.7
    Pälli        Fortum          51.0    14.0
Pyhakoski        Fortum         147.0    32.4
   Montta        Fortum          47.0    12.2
Merikoski Oulun Energia          40.0    11.0


In [33]:
# --- 4.5 Hourly FlexTool inflow bundle ---
# Inputs  : daily sub-catchment inflows from HBV-light (m³/s)
# Method  : piecewise-constant repetition — each daily value is repeated 24 times
#           This preserves daily volumes exactly.
# Units   : m³/s (inflow per FlexTool cascade node), hourly timestep
# Limitation: introduces a step-function discontinuity at each day boundary;
#             this may generate artificial sub-daily regulation signatures in FlexTool output.
#             Document this artefact in the Discussion (relevant to RQ1 and RQ3).

def daily_to_hourly_constant(daily_series: pd.Series) -> pd.Series:
    """
    Piecewise-constant hourly disaggregation (volume-preserving).
    daily_series : daily pd.Series with DatetimeIndex
    Returns      : hourly pd.Series
    """
    hourly_index = pd.date_range(
        daily_series.index[0],
        periods=len(daily_series) * 24,
        freq='h'
    )
    return daily_series.reindex(hourly_index, method='ffill').rename(daily_series.name)

print("daily_to_hourly_constant() defined. Apply to each FlexTool tributary node inflow series.")

daily_to_hourly_constant() defined. Apply to each FlexTool tributary node inflow series.


---
# 5. Export

Write all analysis-ready datasets to `../HydroReg-Oulujoki_analysis_ready/`. Write a `provenance.json` sidecar documenting input files, processing parameters, processing date, and software versions.

This notebook writes to the directories below; the repository's `processed_data/` folder holds only lightweight manifests and checksums.

| Output subfolder | Contents |
|-----------------|----------|
| `spatial/` | Sub-catchment GeoPackage with HRU fractions; clipped CLC GeoTIFF |
| `forcing/` | HBV-light daily forcing `.txt` per sub-catchment; observed Q `.parquet` + `.csv` |
| `flextool_inputs/` | Hourly inflow `.parquet` + `.csv` per cascade node; plant data `.csv` |

In [34]:
# --- 5.1 Export spatial layers ---

# Sub-catchments with HRU fractions
# out_catchments = spatial_dir / "oulujoki_subcatchments_hru.gpkg"
# catchments_hru.to_file(out_catchments, driver='GPKG')
# print(f"Written: {out_catchments} ({out_catchments.stat().st_size / 1e6:.1f} MB)")

# Clipped reclassified CLC raster (written during Section 3.2 processing)
# out_clc = spatial_dir / "clc2018_fi20m_oulujoki_reclass.tif"
# print(f"Written: {out_clc} ({out_clc.stat().st_size / 1e6:.1f} MB)")
pass

In [35]:
# --- 5.2 Export HBV-light forcing and observed discharge ---

# HBV-light forcing: one whitespace-delimited .txt per sub-catchment
# Columns: Date (YYYY-MM-DD)  P (mm/day)  T (°C)  PET (mm/day)
# hbv_dir = forcing_dir / "hbv_light"
# hbv_dir.mkdir(exist_ok=True)
# for catchment_id, df in hbv_forcing.items():
#     out = hbv_dir / f"forcing_{catchment_id}.txt"
#     df.to_csv(out, sep='\t', index=True, float_format='%.3f')

# Observed discharge (calibration / validation target)
# q_obs_out = forcing_dir / "discharge_obs_daily.parquet"
# q_obs.to_parquet(q_obs_out)
# q_obs.to_csv(q_obs_out.with_suffix('.csv'))

# Plant operational envelope
# PLANT_DATA.to_csv(flextool_dir / "plant_characteristics.csv", index=False)
pass

In [36]:
# --- 5.3 Export hourly FlexTool inflow bundles ---

# One file per cascade node
# for node_id, series in flextool_inflows.items():
#     out = flextool_dir / f"inflow_{node_id}_hourly.parquet"
#     series.to_frame(name='Q_m3s').to_parquet(out)
#     series.to_frame(name='Q_m3s').to_csv(out.with_suffix('.csv'))
#     print(f"Written: {out.name}")
pass

In [39]:
# --- 5.4 Provenance sidecar ---
provenance = {
    'created_utc'            : datetime.utcnow().isoformat() + 'Z',
    'notebook'               : 'SE2_data_processing.ipynb',
    'se1_notebook'           : 'SE1_data_access.ipynb',
    'project'                : 'HydroReg-Oulujoki',
    'target_crs'             : TARGET_CRS,
    'study_period'           : {'start': START_DATE, 'end': END_DATE},
    'python_version'         : sys.version,
    'key_package_versions'   : {
        'geopandas' : gpd.__version__,
        'pandas'    : pd.__version__,
        'numpy'     : np.__version__,
        'rasterio'  : rasterio.__version__,
        'pyproj'    : pyproj.__version__,
    },
    'raw_input_files'        : [
        str(direct_dir / 'valumaalueet.zip'),
        str(direct_dir / 'clc2018_fi20m.zip'),
        str(direct_dir / 'clc2018eu25ha.zip'),
        str(api_dir),
    ],
    'pet_method'             : 'Oudin (2005) doi:10.1016/j.jhydrol.2004.08.026',
    'met_spatial_interp'     : 'Thiessen polygon area-weighted average',
    'hourly_disagg_method'   : 'piecewise-constant repetition (volume-preserving)',
    'turbine_efficiency'     : TURBINE_EFFICIENCY,
    'gap_interp_max_days'    : GAP_INTERP_MAX_DAYS,
    'clc_reclass_map'        : {k: list(v) for k, v in CLC_RECLASS.items()},
    'notes'                  : 'Update raw_input_files with actual SE1 filenames before publishing.'
}

# Save JSON sidecar
prov_path = ar_dir / 'provenance.json'
prov_path.write_text(json.dumps(provenance, indent=2))

# --- Display as table ---
def flatten_provenance(p):
    rows = []
    for k, v in p.items():
        if k == 'key_package_versions':
            for pkg, ver in v.items():
                rows.append({'field': f'  {pkg}', 'value': ver})
        elif k == 'study_period':
            rows.append({'field': k, 'value': f"{v['start']} → {v['end']}"})
        elif k == 'raw_input_files':
            rows.append({'field': k, 'value': '\n'.join(v)})
        elif k == 'clc_reclass_map':
            rows.append({'field': k, 'value': ', '.join(
                f"{cls}: {codes[0]}–{codes[-1]}" for cls, codes in v.items()
            )})
        else:
            rows.append({'field': k, 'value': str(v)})
    return rows

prov_df = pd.DataFrame(flatten_provenance(provenance)).set_index('field')
display(prov_df)

print(f"\nProvenance written to: {prov_path}")

,value
field,
created_utc,2026-03-25T10:16:47.809410Z
notebook,SE2_data_processing.ipynb
se1_notebook,SE1_data_access.ipynb
project,HydroReg-Oulujoki
target_crs,EPSG:3067
study_period,2015-01-01 → 2024-12-31
python_version,"3.11.6 | packaged by conda-forge | (main, Oct ..."
geopandas,1.1.2
pandas,3.0.0



Provenance written to: /home/jovyan/HydroReg-Oulujoki_analysis_ready/provenance.json


In [38]:
# --- 5.5 Output file inventory ---
# Print a summary of all files written to the analysis-ready directory.

files = [
    {'file': str(p.relative_to(ar_dir)), 'size_mb': round(p.stat().st_size / 1e6, 2)}
    for p in ar_dir.rglob('*') if p.is_file()
]
inventory = pd.DataFrame(files).sort_values('file').reset_index(drop=True)
print(f"\nTotal files: {len(inventory)}")
print(f"Total size : {inventory['size_mb'].sum():.1f} MB")
display(inventory)


Total files: 1
Total size : 0.0 MB


,file,size_mb
0,provenance.json,0.01


---
## Need help?

Project issues and discussions: https://github.com/sajjadmv